# B07-03 — 반복 그룹 검증과 라운드 보류 평가 (v1)

**BioCulture | 실제 *E. coli* 배양 자료**

**연구 질문:** B07-02에서 관측한 예측 성능은 새로운 배지 조성 분할에서도 유지되는가? 새로운 실험 라운드에서는 어떻게 달라지는가?

- 입력: B07-00 Clean v2의 1,025개 조성 표, 44개 농도 열, 곡선-라운드 정보; B07-02의 기존 분할 기록
- 목표: `K_median`, `r_median` (원본 평가 지표의 조성별 중앙값이며 원 논문 정의 미확정)
- 방법: 여러 시드의 조성 단위 분할, 학습 데이터 전용 전처리, 라운드 전체 보류와 겹침 제거
- **중요:** 본 실습은 재현 가능한 탐색적 평가입니다. 실험 간 독립성·인과성·새 연구기관으로의 일반화 성능을 보증하지 않습니다.

**실행 환경:** Python 3.11, `numpy`, `pandas`, `scikit-learn`, `matplotlib`, Jupyter. Windows `Python (BioImage AI)` 커널에서 프로젝트의 `notebooks` 폴더에 두고 실행합니다.

## 1. 환경, 입력 승인 상태 및 출력 폴더

B07-00의 조성 일치 검증을 확인합니다. 원본 데이터의 결측치를 삭제·보정하지 않고, 각 목표의 유효값만 모델링에 사용합니다.

In [ ]:
from pathlib import Path
import hashlib, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

cwd=Path.cwd().resolve()
ROOT=next((p for p in (cwd,cwd.parent,cwd/'bioimage-ai') if (p/'results'/'bioculture'/'B07_00_ml_data_audit_v2').is_dir()),None)
if ROOT is None: raise FileNotFoundError('B07_00_ml_data_audit_v2 디렉터리가 없습니다. 프로젝트 루트/ notebooks에서 실행하세요.')
BASE=ROOT/'results'/'bioculture'
A=BASE/'B07_00_ml_data_audit_v2'
B=BASE/'B07_02_medium_growth_ml_v1'
OUT=BASE/'B07_03_group_round_validation_v1'; OUT.mkdir(parents=True,exist_ok=True)
PROFILE=A/'ml_medium_profile_level_audit_table.csv'
FEATURES=A/'medium_feature_audit.csv'
CURVES=A/'ml_curve_level_audit_table.csv'
AUDIT=A/'B07_00_validation_report.json'
PRIOR_SPLIT=B/'medium_profile_train_test_split.csv'
for path in (PROFILE,FEATURES,CURVES,AUDIT,PRIOR_SPLIT):
    if not path.is_file():raise FileNotFoundError(path)
assert json.loads(AUDIT.read_text(encoding='utf-8'))['status']=='COMPOSITION_AUDIT_PASSED_NOT_MODEL_VALIDATION'
print('프로젝트:',ROOT,'\n출력:',OUT)

## 2. 44개 성분과 분석 단위 감사

`medium_profile_id`는 44개 실제 배지 농도로 만든 고유 조성 ID입니다. `Condition ID`, `round`, 기존 목표값, 품질 플래그는 모델 입력에서 제외합니다.

In [ ]:
profile=pd.read_csv(PROFILE,encoding='utf-8-sig',dtype={'medium_profile_id':'string'},low_memory=False)
fa=pd.read_csv(FEATURES,encoding='utf-8-sig')
feature_cols=fa['feature'].astype(str).tolist()
assert len(feature_cols)==44 and len(set(feature_cols))==44
assert len(profile)==1025 and profile.medium_profile_id.is_unique
assert set(feature_cols+['K_median','r_median','medium_profile_id']).issubset(profile.columns)
X=profile.set_index('medium_profile_id')[feature_cols].apply(pd.to_numeric,errors='coerce')
ytab=profile.set_index('medium_profile_id')[['K_median','r_median']].apply(pd.to_numeric,errors='coerce')
assert not X.isna().any().any(), 'B07-00에서는 배지 농도 결측 0건으로 확인됨'
assert X.index.is_unique and X.shape==(1025,44)
meta=pd.read_csv(CURVES,encoding='utf-8-sig',usecols=['medium_profile_id','round'],dtype={'medium_profile_id':'string'})
assert meta.medium_profile_id.notna().all()
assert set(meta.medium_profile_id)==set(X.index)
# 라운드 값은 문자열로 표준화; 수치 1/1.0 형식도 처리
meta['round_key']=meta['round'].map(lambda v: str(int(float(v))) if pd.notna(v) and str(v).replace('.','',1).isdigit() else str(v))
round_sets=meta.groupby('medium_profile_id')['round_key'].agg(lambda s:tuple(sorted(set(s)))).reindex(X.index)
assert round_sets.notna().all()
print('X:',X.shape,'K 유효:',ytab.K_median.notna().sum(),'r 유효:',ytab.r_median.notna().sum())
print('라운드:',sorted(meta.round_key.unique()),'복수 라운드 조성:',int(round_sets.map(len).gt(1).sum()))

## 3. 평가 함수와 고정 모델

B07-02의 탐색적 선택값을 **사전에 고정한 평가 설정**으로 사용합니다 (`Ridge alpha=100`, Random Forest `max_features`는 목표별 1.0/0.5, `min_samples_leaf=2`). 새로운 평가 분할에서 매개변수를 다시 선택하지 않아 시험 데이터의 영향을 피합니다. 학습에는 매번 새로운 Pipeline을 생성합니다. 결과는 B07-02의 모델 선택 과정을 포함한 완전히 독립적인 *nested CV*가 아니므로 그 한계를 명시합니다.

동일한 조성은 한 번의 평가에서 오직 학습 또는 시험 중 한쪽에만 포함됩니다.

In [ ]:
TARGETS=['K_median','r_median']
MODELS=['DummyMedian','Ridge','RandomForest']
SEEDS=[20261019,20261020,20261021,20261022,20261023]
RF_MAX_FEATURES={'K_median':1.0,'r_median':0.5}

def estimator(model,target,seed):
    if model=='DummyMedian': core=DummyRegressor(strategy='median')
    elif model=='Ridge': core=Ridge(alpha=100)
    elif model=='RandomForest': core=RandomForestRegressor(n_estimators=120,max_features=RF_MAX_FEATURES[target],min_samples_leaf=2,n_jobs=-1,random_state=seed)
    else: raise ValueError(model)
    steps=[('imputer',SimpleImputer(strategy='median'))]
    if model=='Ridge': steps.append(('scaler',StandardScaler()))
    steps.append(('model',core))
    return Pipeline(steps)

def fit_eval(train_ids,test_ids,target,model,seed):
    train_ids=[x for x in train_ids if pd.notna(ytab.loc[x,target])]
    test_ids=[x for x in test_ids if pd.notna(ytab.loc[x,target])]
    if set(train_ids)&set(test_ids):raise AssertionError('조성 누출!')
    if len(train_ids)<30 or len(test_ids)<5:
        return {'status':'INSUFFICIENT_TARGET_ROWS','n_train':len(train_ids),'n_test':len(test_ids)},pd.DataFrame()
    est=estimator(model,target,seed)
    est.fit(X.loc[train_ids],ytab.loc[train_ids,target])
    pred=np.asarray(est.predict(X.loc[test_ids]),dtype=float)
    true=ytab.loc[test_ids,target].to_numpy(dtype=float)
    result={'status':'EVALUATED','n_train':len(train_ids),'n_test':len(test_ids),
            'MAE':float(mean_absolute_error(true,pred)),
            'RMSE':float(np.sqrt(mean_squared_error(true,pred))),
            'R2':float(r2_score(true,pred)) if len(true)>=2 else np.nan}
    details=pd.DataFrame({'medium_profile_id':test_ids,'y_true':true,'y_pred':pred,'residual':true-pred})
    return result,details

# 독립된 간단한 합성 테스트: 훈련·시험 분리, 출력 및 유한 수치 확인
_tmp_ids=list(X.index[:45]); _res,_pred=fit_eval(_tmp_ids[:35],_tmp_ids[35:],'K_median','DummyMedian',0)
assert _res['status']=='EVALUATED' and len(_pred)>=5
print('모델 평가 함수 검사 통과')

## 4. 고유 조성별 반복 Holdout (5개 분할)

서로 다른 고정 시드를 사용해 80:20 조성 단위 분할을 반복합니다. 각 시드에서 두 목표의 유효값이 서로 달라 실제 평가 수가 다를 수 있습니다. 이때 결과 변동은 분할 민감도의 탐색적 측정이지 독립 실험의 신뢰구간이 아닙니다.

In [ ]:
repeat_rows=[]; repeat_pred=[]; split_rows=[]
ids=X.index.astype(str).tolist()
for seed in SEEDS:
    tr,te=train_test_split(ids,test_size=0.2,random_state=seed,shuffle=True)
    assert set(tr).isdisjoint(te)
    split_rows.extend([{'seed':seed,'medium_profile_id':v,'split':'train'} for v in tr])
    split_rows.extend([{'seed':seed,'medium_profile_id':v,'split':'test'} for v in te])
    for target in TARGETS:
        for model in MODELS:
            res,pred=fit_eval(tr,te,target,model,seed)
            repeat_rows.append({'seed':seed,'target':target,'model':model,**res})
            if not pred.empty:
                pred['seed']=seed;pred['target']=target;pred['model']=model
                repeat_pred.append(pred)
repeat=pd.DataFrame(repeat_rows); repeat_predictions=pd.concat(repeat_pred,ignore_index=True)
split_log=pd.DataFrame(split_rows)
assert len(repeat)==len(SEEDS)*len(TARGETS)*len(MODELS)
assert (repeat.status=='EVALUATED').all()
assert (split_log.groupby(['seed','medium_profile_id']).size()==1).all()
repeat.to_csv(OUT/'repeated_group_holdout_metrics.csv',index=False,encoding='utf-8-sig')
repeat_predictions.to_csv(OUT/'repeated_group_holdout_predictions.csv',index=False,encoding='utf-8-sig')
split_log.to_csv(OUT/'repeated_group_holdout_split_manifest.csv',index=False,encoding='utf-8-sig')
summary=repeat.groupby(['target','model']).agg(n_splits=('seed','size'),MAE_mean=('MAE','mean'),MAE_sd=('MAE','std'),R2_mean=('R2','mean'),R2_sd=('R2','std'),R2_min=('R2','min'),R2_max=('R2','max')).reset_index()
summary.to_csv(OUT/'repeated_group_holdout_summary.csv',index=False,encoding='utf-8-sig')
display(summary)

## 5. 라운드 전체 보류 평가 설계

관측된 라운드 하나를 시험 대상으로 삼습니다. **학습에서는 그 라운드를 한 번이라도 포함한 모든 조성을 완전히 제거**하고, 시험에는 해당 라운드에만 나타난 *단일 라운드 조성*을 사용합니다. 여러 라운드에 걸친 조성은 경계 자료로 따로 기록하고 시험 성능에는 넣지 않습니다. 이렇게 하면 같은 조성이 양쪽에 들어가는 누출을 방지합니다.

단, 각 라운드마다 배지 종류 자체가 달라 배지 조성 변화와 라운드 효과를 분리할 수 없습니다. 이 결과는 **분포 이동 스트레스 테스트**, 순수한 실험 장비/일자의 효과 측정이 아닙니다.

In [ ]:
rounds=sorted(meta.round_key.unique())
round_manifest=[]; round_rows=[]; round_preds=[]
for r in rounds:
    seen=[pid for pid,rs in round_sets.items() if r in rs]
    pure=[pid for pid in seen if len(round_sets.loc[pid])==1]
    mult=[pid for pid in seen if len(round_sets.loc[pid])>1]
    train=[pid for pid,rs in round_sets.items() if r not in rs]
    assert not set(train)&set(seen)
    assert not set(train)&set(pure)
    round_manifest.append({'held_out_round':r,'n_train_profiles':len(train),
                           'n_pure_test_profiles':len(pure),'n_multiround_excluded':len(mult),
                           'n_profiles_touching_heldout_round':len(seen),
                           'n_other_round_profiles_in_training':len(train)})
    for target in TARGETS:
        for model in MODELS:
            res,pred=fit_eval(train,pure,target,model,seed=20261019)
            round_rows.append({'held_out_round':r,'target':target,'model':model,**res})
            if not pred.empty:
                pred['held_out_round']=r;pred['target']=target;pred['model']=model
                round_preds.append(pred)
round_design=pd.DataFrame(round_manifest); round_eval=pd.DataFrame(round_rows)
round_predictions=pd.concat(round_preds,ignore_index=True) if round_preds else pd.DataFrame()
round_design.to_csv(OUT/'leave_one_round_out_design.csv',index=False,encoding='utf-8-sig')
round_eval.to_csv(OUT/'leave_one_round_out_metrics.csv',index=False,encoding='utf-8-sig')
round_predictions.to_csv(OUT/'leave_one_round_out_predictions.csv',index=False,encoding='utf-8-sig')
print('라운드별 보류 설계:');display(round_design)
print('시험 가능한 성능:');display(round_eval[round_eval.model=='RandomForest'])
assert (round_design.n_train_profiles+round_design.n_profiles_touching_heldout_round==len(X)).all()
assert (round_design.n_pure_test_profiles+round_design.n_multiround_excluded==round_design.n_profiles_touching_heldout_round).all()

## 6. 반복 분할과 라운드 보류의 결과 비교·시각화

아래 그림은 분할 결과의 변동과 새 라운드 스트레스 테스트를 구별하여 보여줍니다. 서로 다른 데이터 분포이므로 같은 해석으로 단순 비교하지 않습니다.

In [ ]:
fig,axes=plt.subplots(1,2,figsize=(12,4))
for model in MODELS:
    sub=repeat[repeat.model==model]
    for target in TARGETS:
        q=sub[sub.target==target]
        axes[0].plot(range(len(q)),q.R2.values,marker='o',label=f'{model} {target}')
axes[0].set_xticks(range(len(SEEDS)),SEEDS,rotation=45)
axes[0].set(ylabel='R²',xlabel='Random seed',title='Repeated composition-disjoint holdout')
axes[0].axhline(0,color='gray',lw=.8);axes[0].legend(fontsize=7)
for target in TARGETS:
    q=round_eval[(round_eval.model=='RandomForest')&(round_eval.target==target)&(round_eval.status=='EVALUATED')]
    axes[1].plot(q.held_out_round,q.R2,marker='o',label=target)
axes[1].axhline(0,color='gray',lw=.8)
axes[1].set(xlabel='Held-out round',ylabel='R²',title='Leave-one-round-out (pure profiles)')
axes[1].legend()
fig.tight_layout()
fig.savefig(OUT/'group_vs_round_validation.png',dpi=160)
plt.show()
print('반복 분할 요약:');display(summary.round(4))

## 7. 재현성, 데이터 누출 및 산출물 감사

결과 파일을 다시 읽고 고유 조성 분할이 유지되는지 검사합니다. 파일 SHA256은 입력 및 출력 바이트에 대한 추적용이며 모델의 생물학적 타당성 검사는 아닙니다.

In [ ]:
reloaded=pd.read_csv(OUT/'repeated_group_holdout_metrics.csv',encoding='utf-8-sig')
assert len(reloaded)==len(SEEDS)*len(TARGETS)*len(MODELS)
check=pd.read_csv(OUT/'repeated_group_holdout_split_manifest.csv',encoding='utf-8-sig',dtype={'medium_profile_id':'string'})
assert len(check)==len(SEEDS)*len(X)
assert not check.duplicated(['seed','medium_profile_id']).any()
assert set(check.split)=={'train','test'}
assert pd.read_csv(OUT/'leave_one_round_out_design.csv',encoding='utf-8-sig').shape[0]==len(rounds)

def sha(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as f:
        for chunk in iter(lambda:f.read(1024*1024),b''):h.update(chunk)
    return h.hexdigest()
report={'status':'REPEATED_GROUP_AND_ROUND_SHIFT_EVALUATED_NOT_CAUSAL',
        'n_profiles':len(X),'n_features':len(feature_cols),'targets':TARGETS,
        'repeated_split_seeds':SEEDS,'n_repeat_evaluations':len(repeat),
        'n_rounds':len(rounds),'n_round_evaluations':int((round_eval.status=='EVALUATED').sum()),
        'split_disjointness':'verified by medium_profile_id',
        'round_strategy':'train: profiles not in held-out round; test: pure single-round profiles; multiround excluded',
        'hyperparameters':'fixed from exploratory B07-02 choice; not independent nested selection',
        'limitations':['Rounds and composition distributions are confounded',
                       'K/r original definitions and units not fully established',
                       'Model outcomes are associations, not causal effects',
                       'Repeated split variability is not an independent-study confidence interval']}
(OUT/'B07_03_validation_report.json').write_text(json.dumps(report,ensure_ascii=False,indent=2),encoding='utf-8')
items=[('input',p) for p in (PROFILE,FEATURES,CURVES,AUDIT,PRIOR_SPLIT)]
items += [('output',p) for p in OUT.iterdir() if p.is_file() and p.name!='file_manifest_sha256.csv']
manifest=pd.DataFrame([{'kind':kind,'file':str(p.relative_to(ROOT)),'bytes':p.stat().st_size,'sha256':sha(p)} for kind,p in sorted(items,key=lambda x:str(x[1]))])
manifest.to_csv(OUT/'file_manifest_sha256.csv',index=False,encoding='utf-8-sig')
print('재검증 통과. 산출물:',len(list(OUT.iterdir())))
print(json.dumps(report,ensure_ascii=False,indent=2))

## 8. 정리 및 연습문제

**이번 Notebook이 검증하는 것:** 동일한 배지 조성 간 누출이 없는 반복 Holdout에서의 예측 성능 변동, 그리고 특정 라운드의 순수 조성을 보류했을 때의 분포 이동 민감도입니다.

**검증하지 않는 것:** 라운드 변화의 인과적 효과, 완전히 새로운 실험실에 대한 성능, `K`와 `r`의 확정적인 생물학적 정의.

### 연습문제 (정답과 Python 풀이는 후속 Word 해설서에 수록)

1. `medium_profile_id`를 기준으로 분할할 때 곡선 단위 무작위 분할보다 데이터 누출이 적은 이유를 설명하세요.
2. 반복 분할에서 Random Forest의 `R²` 최솟값과 최댓값을 두 목표별로 계산하세요. 평균만 제시했을 때의 위험을 설명하세요.
3. 임의의 한 라운드에서 `n_multiround_excluded`가 의미하는 바를 설명하고 이를 학습·시험에 섞었을 때의 문제를 쓰세요.
4. 특정 라운드 보류의 `R²`가 음수라면 무엇을 의미하나요? 라운드 교란과 조성 분포 이동의 차이를 논하세요.
5. 원본 `K`, `r` 대신 B06에서 계산한 다른 목표 지표를 사용하는 경우 변경할 코드와 새로 필요한 품질 검사를 제안하세요.

**다음 단계:** 사용자 PC의 실제 실행 결과를 검토하고 B07-03 Word 해설서에 연습문제 전체 풀이를 작성합니다. 이후 B07-04 모델 해석·불확실성 단계로 진행합니다.